In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from imblearn.over_sampling import SMOTE
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, classification_report)

X_train = pd.read_csv('../data/processed/X_train.csv')
X_test = pd.read_csv('../data/processed/X_test.csv')
y_train = pd.read_csv('../data/processed/y_train.csv').squeeze()
y_test = pd.read_csv('../data/processed/y_test.csv').squeeze()

print('Original training fraud rate:', y_train.mean())

Original training fraud rate: 0.0169375


In [2]:
# Apply SMOTE to the training data only
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print(f'Before SMOTE: {len(y_train):,} rows, {y_train.mean()*100:.2f}% fraud')
print(f'After SMOTE:  {len(y_train_smote):,} rows, {y_train_smote.mean()*100:.2f}% fraud')

Before SMOTE: 16,000 rows, 1.69% fraud
After SMOTE:  31,458 rows, 50.00% fraud


In [3]:
# Reusable evaluation function
def evaluate(name, model, X_test, y_test):
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1] if hasattr(model, 'predict_proba') else None
    result = {
        'model': name,
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred, zero_division=0),
        'recall': recall_score(y_test, y_pred, zero_division=0),
        'f1': f1_score(y_test, y_pred, zero_division=0),
        'roc_auc': roc_auc_score(y_test, y_proba) if y_proba is not None else None
    }
    return result

In [5]:
# Retrain all 4 models on SMOTE balanced data
results_smote = []

models_smote = {
    'Logistic Regression (SMOTE)': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree (SMOTE)': DecisionTreeClassifier(random_state=42),
    'KNN (SMOTE)': KNeighborsClassifier(n_neighbors=5),
    'SVM (SMOTE)': SVC(kernel='rbf', probability=True, random_state=42)
}

for name, model in models_smote.items():
    model.fit(X_train_smote, y_train_smote)
    results_smote.append(evaluate(name, model, X_test, y_test))

results_smote_df = pd.DataFrame(results_smote).sort_values('f1', ascending=False)
results_smote_df

,model,accuracy,precision,recall,f1,roc_auc
0,Logistic Regression (SMOTE),0.89200,0.114407,0.794118,0.200000,0.930697
3,SVM (SMOTE),0.97900,0.264706,0.132353,0.176471,0.650429
2,KNN (SMOTE),0.91875,0.089457,0.411765,0.146982,0.701727
1,Decision Tree (SMOTE),0.96750,0.092105,0.102941,0.097222,0.542696


In [6]:
#Retraining using class_weight='balanced' instead (no resampling, original imbalanced data)
results_cw = []

models_cw = {
    'Logistic Regression (class_weight)': LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
    'Decision Tree (class_weight)': DecisionTreeClassifier(class_weight='balanced', random_state=42),
    'SVM (class_weight)': SVC(kernel='rbf', probability=True, class_weight='balanced', random_state=42)
}
# Note: KNN has no class_weight parameter — it's excluded here, that's expected

for name, model in models_cw.items():
    model.fit(X_train, y_train)
    results_cw.append(evaluate(name, model, X_test, y_test))

results_cw_df = pd.DataFrame(results_cw).sort_values('f1', ascending=False)
results_cw_df

,model,accuracy,precision,recall,f1,roc_auc
2,SVM (class_weight),0.97900,0.326087,0.220588,0.263158,0.896068
0,Logistic Regression (class_weight),0.88250,0.107422,0.808824,0.189655,0.934429
1,Decision Tree (class_weight),0.97125,0.101695,0.088235,0.094488,0.537378
